# RetailMind AI — Day 13: BERT for review sentiment and ticket routing

Does DistilBERT (a small BERT that has already read a lot of text) read reviews better than the Day 12 word-counting baseline? We try it frozen and fine-tuned.

Training takes about 30 minutes, so this notebook **reads the saved results** from `python -m src.train_day13`. It then tries the saved models on new text; the two BERT models need `pip install torch transformers`.

In [1]:
# Work from the project root so the `src` package can be imported
import os, sys
ROOT = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
os.chdir(ROOT)
sys.path.insert(0, ROOT)
print("project folder:", os.path.basename(ROOT))

project folder: day3-4


## 1. Sentiment: three models, same 1,000 test reviews

In [2]:
import json
day13 = json.load(open("models/day13_results.json"))
print(f"{day13['train_reviews']} training reviews, {day13['test_reviews']} test reviews | model: {day13['model']}\n")
print(f"{'Model':<28}{'Accuracy':>10}{'F1':>9}")
for name, s in day13["sentiment"].items():
    print(f"{name:<28}{s['accuracy']:>10}{s['f1']:>9}")

4000 training reviews, 1000 test reviews | model: distilbert/distilbert-base-uncased

Model                         Accuracy       F1
TF-IDF + LR (Day 12)              0.87   0.8745
DistilBERT frozen + LR           0.867   0.8722
DistilBERT fine-tuned            0.902   0.9063


## 2. Is the lift real? (paired bootstrap)

The test set is only 1,000 reviews, so we re-sample it 2,000 times. If the 95% range of the difference includes 0, it could be luck.

In [3]:
for name, d in day13["lift_over_tfidf"].items():
    verdict = "above 0: likely real" if d["range_95"][0] > 0 else "includes 0: could be luck"
    print(f"{name:<26} {d['accuracy_minus_tfidf']:+.3f} against TF-IDF, 95% range {d['range_95']}  -> {verdict}")

DistilBERT frozen + LR     -0.003 against TF-IDF, 95% range [-0.026, 0.02]  -> includes 0: could be luck
DistilBERT fine-tuned      +0.032 against TF-IDF, 95% range [0.011, 0.054]  -> above 0: likely real


## 3. What does it cost?

In [4]:
c = day13["cost"]
print(f"speed: TF-IDF {c['tfidf_reviews_per_second']:,} reviews/s | DistilBERT {c['bert_reviews_per_second']} reviews/s  (about {c['tfidf_reviews_per_second'] / c['bert_reviews_per_second']:.0f}x slower)")
print(f"size:  TF-IDF {c['tfidf_file_mb']} MB | DistilBERT {c['bert_download_mb']} MB  (about {c['bert_download_mb'] / c['tfidf_file_mb']:.0f}x bigger)")

speed: TF-IDF 34,602 reviews/s | DistilBERT 112.9 reviews/s  (about 306x slower)
size:  TF-IDF 1.2 MB | DistilBERT 268 MB  (about 223x bigger)


## 4. Support-ticket routing

Tickets were made from reviews by trigger words (which were then removed), 500 per category. Guessing gives 0.25.

In [5]:
t = day13["tickets"]
print("categories:", t["categories"], "| train", t["train"], "| test", t["test"])
for name in ("TF-IDF + LR", "DistilBERT frozen + LR"):
    print(f"{name:<24} accuracy {t[name]['accuracy']}  recall per category {t[name]['recall_per_category']}")
print("BERT minus TF-IDF, 95% range:", t["difference_range_95"], "(includes 0: no lift)")

categories: ['how-to', 'quality', 'refund', 'shipping'] | train 1600 | test 400
TF-IDF + LR              accuracy 0.4925  recall per category {'how-to': 0.602, 'quality': 0.438, 'refund': 0.429, 'shipping': 0.511}
DistilBERT frozen + LR   accuracy 0.455  recall per category {'how-to': 0.592, 'quality': 0.384, 'refund': 0.439, 'shipping': 0.413}
BERT minus TF-IDF, 95% range: [-0.0925, 0.02] (includes 0: no lift)


## 5. Try the text module on new reviews (TF-IDF needs no big download)

In [6]:
from src.sentiment_service import predict_sentiment

reviews = ["Absolutely love it, works perfectly and arrived early.",
           "Total waste of money. Stopped working after two days.",
           "It is not bad at all, quite good actually."]
for review, answer in zip(reviews, predict_sentiment(reviews, backend="tfidf")):
    print(f"{answer['sentiment']:<9} (P positive {answer['probability_positive']:.2f})  {review}")

positive  (P positive 0.98)  Absolutely love it, works perfectly and arrived early.
negative  (P positive 0.04)  Total waste of money. Stopped working after two days.
negative  (P positive 0.19)  It is not bad at all, quite good actually.


## 6. Conclusion

- **Fine-tuned DistilBERT beats the baseline: 0.902 vs 0.870 (+3.2 points, 95% range +1.1 to +5.4).** **Frozen** DistilBERT ties it (0.867).
- **The cost is large:** about 300 times slower and 220 times bigger. Whether +3.2 points is worth it depends on the job.
- **One fine-tuning run**, so treat 0.902 as "about 0.90". Negation ("not bad") is still hard for every model.
- **Ticket routing shows no BERT lift** (0.455 vs 0.4925) and both are weak, because the labels come from keyword rules with the keywords removed. Do not use it as a real router.